In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import json
import math
import os
from datetime import datetime, timedelta
from datetime import date

PREDICT_DAYS = 21

with open('config.json', 'r') as f:
    config = json.load()
    analysis_config = config['analysis_settings']

print("Configuration loaded successfully.")

input_files = analysis_config.get('input_data_files', [])
all_dataframes = []

print("--- Loading and combining data sources ---")
for file_path in input_files:
    if os.path.exists(file_path):
        print(f"Reading file: {file_path}")
        if os.path.getsize(file_path) > 0:
            all_dataframes.append(pd.read_csv(file_path))
        else:
            print(f"Warning: Data file '{file_path}' is empty. Skipping.")
    else:
        print(f"Warning: Data file '{file_path}' not found. Skipping.")

RAW_DATA = pd.concat(all_dataframes, ignore_index=True)

# --- Data Type Conversion and Preparation (Key Step) ---
# THE FIX: Add format='mixed' to handle different date string styles
RAW_DATA['creation_date'] = pd.to_datetime(RAW_DATA['creation_date'], format='mixed')
RAW_DATA['updated_at'] = pd.to_datetime(RAW_DATA['updated_at'], format='mixed')

HISTORY_DAYS = (datetime.now() - RAW_DATA['creation_date'].min()).days
print(f"Data loading complete. Found {len(RAW_DATA)} total records spanning approximately {HISTORY_DAYS} days.")

# Display a random sample to confirm data format
RAW_DATA.sample(5)

TypeError: load() missing 1 required positional argument: 'fp'

In [ ]:
# --- Define Inflow ---
# Daily count of newly created issues
daily_new = RAW_DATA.set_index('creation_date').resample('D').size()

# --- Define Outflow ---
# Daily count of issues that were closed
# We use 'updated_at' date where the status is 'closed', mirroring the 'changeddate' logic
daily_resolved = RAW_DATA[RAW_DATA['status'] == 'closed'].set_index('updated_at').resample('D').size()

# --- Combine into a Base Measures DataFrame ---
base_measures_df = pd.merge(daily_new.rename('inflow'), daily_resolved.rename('outflow'), 
                            left_index=True, right_index=True, how='outer').fillna(0).astype(int)

# Ensure the date range is continuous from the earliest record to today
all_days_index = pd.date_range(start=base_measures_df.index.min(), end=date.today(), freq='D')
base_measures_df = base_measures_df.reindex(all_days_index, fill_value=0)

# Extract lists for plotting and prediction
daily_new_list = base_measures_df['inflow'].tolist()
daily_resolved_list = base_measures_df['outflow'].tolist()
x_axis_history = list(range(-len(daily_new_list), 0))

print("--- Base Measures (Last 5 Days) ---")
print(base_measures_df.tail())

# --- Visualize Historical Data ---
plt.figure(figsize=(15, 5))
plt.plot(x_axis_history, daily_new_list, label="Historical Inflow", color='blue')
plt.plot(x_axis_history, daily_resolved_list, label="Historical Outflow", color='green', alpha=0.7)
plt.title("Historical Defect Inflow vs. Outflow Trend")
plt.xlabel("Days from Today (0 = Today)")
plt.ylabel("Number of Defects")
plt.legend()
plt.grid(True, linestyle='--', alpha=0.6)
plt.show()

In [ ]:
def der_predicted_defects(daily_history, predict_days, show_weekday_fit=False):
    """
    Predicts future daily defect counts using polynomial regression.
    This function replicates the core logic from the course example.
    """
    hist_by_weekday = [[] for _ in range(7)]
    function_by_weekday = [None] * 7
    
    # 1. Group historical data by weekday (0=Monday, 6=Sunday)
    today = date.today()
    for i, count in enumerate(daily_history):
        # Calculate the actual date of the historical data point
        current_date = today - timedelta(days=(len(daily_history) - i))
        weekday = current_date.weekday()
        hist_by_weekday[weekday].append(count)

    # 2. Fit a 3rd degree polynomial model for each day of the week
    for wd in range(7):
        y = hist_by_weekday[wd]
        x = range(-len(y), 0)
        
        if len(y) < 4: # Need at least 4 points to fit a 3rd degree polynomial
            # If data is insufficient, use the average as a constant prediction
            avg = np.mean(y) if y else 0
            function_by_weekday[wd] = np.poly1d([avg])
        else:
            # Fit a 3rd degree polynomial
            coeffs = np.polyfit(x, y, deg=3)
            function_by_weekday[wd] = np.poly1d(coeffs)

    # 3. Interlace the predictions from the 7 models into a daily sequence
    pred_x = list(range(predict_days))
    pred_y = [0] * predict_days
    
    for i in range(predict_days):
        future_date = today + timedelta(days=i + 1)
        weekday = future_date.weekday()
        # The x-coordinate for prediction is the future "week index"
        # e.g., the first upcoming Monday is week_index 0, the next is 1, etc.
        week_index = len(hist_by_weekday[weekday]) + (i // 7)
        prediction = function_by_weekday[weekday](week_index - len(hist_by_weekday[weekday]))
        pred_y[i] = max(prediction, 0) # Prediction cannot be negative

    # Optional: Show the fit for a specific day (e.g., Wednesday)
    if show_weekday_fit:
        wd = 2 # Wednesday
        y_hist = hist_by_weekday[wd]
        x_hist = range(-len(y_hist), 0)
        
        x_pred_weeks = range(math.ceil(predict_days / 7))
        
        plt.figure(figsize=(10, 4))
        plt.scatter(x_hist, y_hist, label=f"Historical Data (Wednesdays)")
        
        # Plot the fitted curve across history and future
        x_curve = np.linspace(min(x_hist) -1, max(x_pred_weeks), 50)
        y_curve = function_by_weekday[wd](x_curve)
        plt.plot(x_curve, y_curve, label="Polynomial Fit Curve", color='red')
        plt.title("Polynomial Regression Fit for Wednesday's Inflow")
        plt.xlabel("Weeks from History/Future Boundary (0 = Next Week)")
        plt.ylabel("Number of Defects")
        plt.legend()
        plt.grid(True, linestyle='--', alpha=0.6)
        plt.show()
        
    return pred_x, pred_y

# --- Execute Prediction ---
print("--- Generating Inflow Prediction ---")
pnew_x, pnew_y = der_predicted_defects(daily_new_list, PREDICT_DAYS, show_weekday_fit=True)

print(f"\nSuccessfully predicted daily inflow for the next {PREDICT_DAYS} days.")

In [ ]:
# --- Calculate Indicator (Traffic Light) ---
average_predicted_inflow = np.mean(pnew_y)
# Use thresholds from our own project definition, which came from the PM interview
thresholds = analysis_config['inflow_thresholds'] 

indicator_color = 'grey'
if average_predicted_inflow > thresholds['red_alert_gt']:
    indicator_color = 'red'
elif average_predicted_inflow < thresholds['yellow_warning_eq']:
    indicator_color = 'green'
else:
    indicator_color = 'yellow'

print(f"Predicted average daily inflow for the next {PREDICT_DAYS} days: {average_predicted_inflow:.2f}")
print(f"Project Health Indicator: {indicator_color.upper()}")

# --- Final Visualization ---
plt.figure(figsize=(15, 6))

# Plot historical data
plt.plot(x_axis_history, daily_new_list, label="Historical Inflow", color='blue', alpha=0.8)

# Plot predicted data
plt.plot(pnew_x, pnew_y, label="Predicted Inflow", color='red', linestyle='--')

# Overlay a colored span to represent the health indicator
plt.axvspan(pnew_x[0], pnew_x[-1], color=indicator_color, alpha=0.2, label=f"Health Status: {indicator_color.upper()}")


plt.title(f"Defect Inflow: History and Prediction for the Next {PREDICT_DAYS} Days", fontsize=16)
plt.xlabel("Days from Today (0 = Today)")
plt.ylabel("Number of Defects")
plt.legend()
plt.grid(True, linestyle='--', alpha=0.6)
plt.show()

# Plot a separate color block as the final "dashboard"
fig, ax = plt.subplots(figsize=(3, 2))
ax.set_facecolor(indicator_color)
ax.tick_params(axis='both', which='both', bottom=False, top=False, left=False, right=False, 
               labelbottom=False, labelleft=False)
ax.set_title(f"Project Health: {indicator_color.upper()}", fontsize=14)
plt.show()